# Minutes of Meeting Generator — Google Colab (T4 GPU)

Runs the MoM pipeline on a free **T4 GPU**, so `large-v3` transcribes a 10-min file in **~1-2 min** instead of ~1 hour on CPU.

**Before you start:**
1. `Runtime → Change runtime type → Hardware accelerator: → **T4 GPU** → Save`.
2. Have your **Groq API key** ready (https://console.groq.com/keys).
3. On your PC, zip the project folder **excluding `.venv`, `models`, `output`** (just `project.py`, `mom_generator/`, `requirements*.txt`). Name it `MoM_generator.zip`.

Run the cells top to bottom.

## 0. Confirm the GPU is attached

In [ ]:
!nvidia-smi
# You should see a Tesla T4. If this errors, you did not select the T4 runtime (see step 1 above).

## 1. Get the code

**Option A (default):** upload the `MoM_generator.zip` you made above.

**Option B (later, once you've pushed to GitHub):** comment out the upload lines and use the `git clone` line instead.

In [ ]:
# --- Option A: upload a zip of the project ---
from google.colab import files
import zipfile, os

uploaded = files.upload()                       # pick MoM_generator.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content')

# Move into the project folder (the one containing project.py).
for root, _dirs, fnames in os.walk('/content'):
    if 'project.py' in fnames:
        os.chdir(root)
        break
print('Working directory:', os.getcwd())

# --- Option B: clone from GitHub (use after you push your changes) ---
# !git clone https://github.com/<you>/MoM_generator.git
# %cd MoM_generator

## 2. Install dependencies (+ GPU libraries)

Colab already ships PyTorch, so we don't reinstall it. The last line pulls the **cuDNN / cuBLAS** wheels that CTranslate2 (faster-whisper's GPU engine) needs.

In [ ]:
!pip -q install faster-whisper groq python-docx python-dotenv scikit-learn librosa webrtcvad-wheels
!pip -q install resemblyzer --no-deps   # --no-deps: avoid its webrtcvad source build
!pip -q install nvidia-cublas-cu12 nvidia-cudnn-cu12  # CTranslate2 GPU runtime libs

## 3. Fix the cuDNN library path (the #1 Colab GPU gotcha)

CTranslate2 loads `libcudnn` at runtime and Colab doesn't have it on the search path by default — the symptom is an error like `Unable to load libcudnn_ops.so`. This points the loader at the wheels we just installed. (Setting `os.environ` here is inherited by the `!python` run below.)

In [ ]:
import os, glob
nvidia_lib_dirs = sorted(set(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/*/lib')))
os.environ['LD_LIBRARY_PATH'] = ':'.join(nvidia_lib_dirs) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
print('LD_LIBRARY_PATH =', os.environ['LD_LIBRARY_PATH'])

# Quick check that the GPU is visible to CTranslate2 itself.
import ctranslate2
print('CTranslate2 CUDA devices:', ctranslate2.get_cuda_device_count(), '(should be 1)')

## 4. Set your Groq API key

Paste it when prompted (hidden input). Don't hard-code the key in the notebook.

In [ ]:
import os, getpass
os.environ['GROQ_API_KEY'] = getpass.getpass('Groq API key: ')
print('GROQ_API_KEY set:', bool(os.environ.get('GROQ_API_KEY')))

## 5. Upload your meeting audio

In [ ]:
from google.colab import files
audio = files.upload()                # pick IITH_10min.wav (or any .mp3/.wav/.m4a)
AUDIO_PATH = next(iter(audio))
print('Uploaded:', AUDIO_PATH)

## 6. Run the pipeline

Clean Tier-1 test: **large-v3**, **English forced**, **voice diarization with 4 speakers**, **no `--vocab`** (so we keep isolating Tier 1). `transcribe_audio` auto-detects the GPU — watch for `[transcriber] Device: cuda`.

In [ ]:
!python project.py "$AUDIO_PATH" \
    --model large-v3 \
    --language en \
    --speakers voice --num-speakers 4 \
    --save-transcript \
    --output output/IITH_colab_tier1_v3_voice.docx

## 7. Download the results

In [ ]:
from google.colab import files
files.download('output/IITH_colab_tier1_v3_voice_transcript.txt')
files.download('output/IITH_colab_tier1_v3_voice.docx')